# 🛸 Antigravity Agent Lite (Maintenance & Bootstrap Tool)

Phiên bản **tối giản** chỉ giữ lại các tác vụ cốt lõi nhất để tương tác với Google Antigravity Agent & Sandbox.
- **Mục đích:** Khi Google cập nhật `google-genai` SDK, `api_version`, tên Agent hoặc cấu trúc `interactions` API, bạn chỉ cần sửa vài dòng ở **Cell 2 & Cell 3**, sau đó dùng chính notebook này để đọc và cập nhật lại `app.py` của bản **Google_Antigravity_Agent_Studio_Pro_v3.5**.

In [ ]:
# === CELL 1: CÀI ĐẶT & NẠP THƯ VIỆN ===
!pip install -q -U google-genai requests python-dotenv

import os
import time
import mimetypes
import requests
from getpass import getpass
from google import genai

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

print("✅ Đã nạp thư viện thành công!")

In [ ]:
# === CELL 2: CẤU HÌNH TRUNG TÂM (CHỖ CHÍNH ĐỂ SỬA KHI GOOGLE CẬP NHẬT) ===

# 1. Lấy API Key (ưu tiên Colab Secrets -> biến môi trường/.env -> nhập tay)
API_KEY = ""
try:
    from google.colab import userdata
    API_KEY = userdata.get("GEMINI_API_KEY") or ""
except Exception:
    pass

if not API_KEY:
    API_KEY = os.environ.get("GEMINI_API_KEY", "").strip()

if not API_KEY:
    API_KEY = getpass("🔑 Nhập GEMINI_API_KEY: ").strip()

# 2. Tham số API / Agent (dễ dàng cập nhật khi Google ra bản mới)
API_VERSION = "v1beta"
AGENT_NAME = "antigravity-preview-09-2026"
USE_BACKGROUND = True      # Nên bật True để tránh timeout khi Agent sửa file lớn (app.py)
POLL_INTERVAL = 3.0        # Giây giữa mỗi lần kiểm tra tiến độ
TIMEOUT = 600.0            # Thời gian chờ tối đa (giây)

SYSTEM_INSTRUCTION = (
    "Bạn là kỹ sư phần mềm chuyên sâu về Python, Gradio và Google GenAI SDK. "
    "Hãy đọc kỹ mã nguồn được cung cấp trong sandbox, phân tích chính xác và hỗ trợ chỉnh sửa/cập nhật theo yêu cầu."
)

# 3. Khởi tạo Client
client = genai.Client(api_key=API_KEY, http_options={"api_version": API_VERSION})
print(f"✅ Đã khởi tạo Client | API: {API_VERSION} | Agent: {AGENT_NAME}")

In [ ]:
# === CELL 3: LÕI GIAO TIẾP AGENT & SANDBOX ===

session_state = {
    "env_id": None,
    "last_interaction_id": None,
}

def extract_text(interaction) -> str:
    if not interaction:
        return ""
    out_txt = getattr(interaction, "output_text", None)
    if out_txt and str(out_txt).strip():
        return str(out_txt).strip()

    texts = []
    for step in (getattr(interaction, "steps", None) or []):
        step_type = getattr(step, "type", "") or step.__class__.__name__
        if step_type == "model_output" or "ModelOutput" in step_type:
            for item in (getattr(step, "content", None) or []):
                t = getattr(item, "text", None)
                if t and str(t).strip():
                    texts.append(str(t))

    if not texts:
        for item in (getattr(interaction, "outputs", None) or []):
            t = getattr(item, "text", None)
            if t and str(t).strip():
                texts.append(str(t))
    return "\n\n".join(texts).strip()

def extract_errors(interaction) -> str:
    errs = getattr(interaction, "errors", None)
    if errs:
        return "; ".join(str(getattr(e, "message", e)) for e in errs)
    return str(getattr(interaction, "error", None) or "Không xác định")

def _build_inline_sources(file_paths):
    sources = []
    for p in (file_paths or []):
        if os.path.isfile(p):
            with open(p, "r", encoding="utf-8", errors="replace") as f:
                content = f.read()
            sources.append({
                "type": "inline",
                "target": os.path.basename(p),
                "content": content,
            })
        else:
            print(f"⚠️ Không tìm thấy file để đính kèm: {p}")
    return sources

def upload_to_sandbox(local_path: str, remote_path: str = None) -> bool:
    if not session_state["env_id"]:
        print("⚠️ Chưa có Sandbox. Hãy gửi ít nhất 1 tin nhắn trước để tạo Sandbox.")
        return False
    if not os.path.isfile(local_path):
        print(f"⚠️ Không tìm thấy file local: {local_path}")
        return False

    clean_env = session_state["env_id"].replace("environments/", "").strip()
    clean_path = (remote_path or os.path.basename(local_path)).lstrip("/")
    mime_type = mimetypes.guess_type(clean_path)[0] or "application/octet-stream"
    with open(local_path, "rb") as f:
        data_bytes = f.read()

    # 1. Thử qua SDK
    try:
        if hasattr(client, "environments") and hasattr(client.environments, "files"):
            client.environments.files.upload(
                environment=clean_env,
                path=clean_path,
                file=data_bytes,
                mime_type=mime_type,
                overwrite=True,
            )
            print(f"📤 Đã upload lên Sandbox: `{clean_path}` ({len(data_bytes):,} bytes)")
            return True
    except Exception as e:
        print(f"ℹ️ SDK upload fallback sang REST ({e})")

    # 2. Fallback REST API
    url = f"https://generativelanguage.googleapis.com/upload/{API_VERSION}/environments/{clean_env}/files/{clean_path}?overwrite=true"
    res = requests.put(url, headers={"x-goog-api-key": API_KEY, "Content-Type": mime_type}, data=data_bytes, timeout=60)
    if res.status_code in (200, 201):
        print(f"📤 Đã upload lên Sandbox (REST): `{clean_path}` ({len(data_bytes):,} bytes)")
        return True
    print(f"❌ Lỗi upload ({res.status_code}): {res.text}")
    return False

def list_sandbox_files():
    if not session_state["env_id"]:
        print("⚠️ Chưa có Sandbox nào đang hoạt động.")
        return []
    clean_env = session_state["env_id"].replace("environments/", "").strip()

    files_list = []
    try:
        resp = client.environments.files.list(environment=clean_env, path="", recursive=True)
        for f in (getattr(resp, "files", None) or []):
            files_list.append(getattr(f, "path", None) or getattr(f, "name", None))
    except Exception:
        url = f"https://generativelanguage.googleapis.com/{API_VERSION}/environments/{clean_env}/files"
        res = requests.get(url, headers={"x-goog-api-key": API_KEY}, timeout=15)
        if res.status_code == 200:
            files_list = [f.get("path") or f.get("name") for f in res.json().get("files", [])]
        else:
            print(f"❌ Lỗi liệt kê file ({res.status_code}): {res.text}")
            return []

    print(f"📂 Tệp trong Sandbox `{clean_env}` ({len(files_list)} mục):")
    for p in files_list:
        print(f"  - {p}")
    return files_list

def download_from_sandbox(remote_path: str, local_path: str = None) -> bool:
    if not session_state["env_id"]:
        print("⚠️ Chưa có Sandbox nào đang hoạt động.")
        return False
    clean_env = session_state["env_id"].replace("environments/", "").strip()
    clean_path = remote_path.lstrip("/")
    save_to = local_path or os.path.basename(clean_path) or "downloaded_file"

    data = None
    try:
        data = client.environments.files.download(environment=clean_env, path=clean_path)
    except Exception:
        url = f"https://generativelanguage.googleapis.com/{API_VERSION}/environments/{clean_env}/files/{clean_path}?alt=media"
        res = requests.get(url, headers={"x-goog-api-key": API_KEY}, timeout=60)
        if res.status_code == 200:
            data = res.content
        else:
            print(f"❌ Lỗi tải file ({res.status_code}): {res.text}")
            return False

    os.makedirs(os.path.dirname(os.path.abspath(save_to)), exist_ok=True)
    with open(save_to, "wb") as f:
        f.write(data)
    print(f"📥 Đã tải `{clean_path}` -> `{save_to}` ({len(data):,} bytes)")
    return True

def ask_agent(prompt: str, files: list = None, reset: bool = False) -> str:
    if reset:
        session_state["env_id"] = None
        session_state["last_interaction_id"] = None
        print("🔄 Đã reset phiên & Sandbox.")

    kwargs = {
        "agent": AGENT_NAME,
        "input": prompt,
    }
    if SYSTEM_INSTRUCTION.strip():
        kwargs["system_instruction"] = SYSTEM_INSTRUCTION.strip()
    if USE_BACKGROUND:
        kwargs["background"] = True
    if session_state["last_interaction_id"]:
        kwargs["previous_interaction_id"] = session_state["last_interaction_id"]

    clean_id = (session_state["env_id"] or "").replace("environments/", "").strip()
    if not clean_id:
        sources = _build_inline_sources(files)
        kwargs["environment"] = {"type": "remote", "sources": sources} if sources else "remote"
    else:
        kwargs["environment"] = {"type": "remote", "environment_id": clean_id}
        if files:
            for fp in files:
                upload_to_sandbox(fp)

    try:
        inter = client.interactions.create(**kwargs)
    except Exception as e:
        if clean_id and ("404" in str(e) or "not found" in str(e).lower()):
            print("⚠️ Sandbox cũ đã hết hạn, đang tự tạo Sandbox mới...")
            session_state["env_id"] = None
            session_state["last_interaction_id"] = None
            kwargs.pop("previous_interaction_id", None)
            sources = _build_inline_sources(files)
            kwargs["environment"] = {"type": "remote", "sources": sources} if sources else "remote"
            inter = client.interactions.create(**kwargs)
        else:
            raise

    if getattr(inter, "environment_id", None):
        session_state["env_id"] = inter.environment_id
    if getattr(inter, "id", None):
        session_state["last_interaction_id"] = inter.id

    # Polling nếu chạy background
    start_t = time.time()
    while getattr(inter, "status", None) in ("in_progress", "queued"):
        if time.time() - start_t > TIMEOUT:
            raise TimeoutError(f"Quá thời gian chờ ({TIMEOUT}s) cho interaction {inter.id}")
        time.sleep(POLL_INTERVAL)
        inter = client.interactions.get(id=inter.id)
        if getattr(inter, "environment_id", None):
            session_state["env_id"] = inter.environment_id

    if getattr(inter, "status", None) == "failed":
        return f"❌ Interaction thất bại: {extract_errors(inter)}"

    return extract_text(inter) or f"⚠️ Không có nội dung text trả về (status={getattr(inter, 'status', None)})."

print("✅ Đã nạp các hàm lõi: ask_agent, upload_to_sandbox, list_sandbox_files, download_from_sandbox")

In [ ]:
# === CELL 4: CHUẨN BỊ FILE DỰ ÁN PRO v3.5 ĐỂ ĐƯA VÀO SANDBOX ===
# Tự động tìm app.py & requirements.txt ở thư mục cha (khi chạy trong notebooks/) hoặc thư mục hiện tại

candidate_files = []
for rel_path in ["../app.py", "app.py", "../requirements.txt", "requirements.txt"]:
    if os.path.isfile(rel_path) and os.path.basename(rel_path) not in [os.path.basename(p) for p in candidate_files]:
        candidate_files.append(rel_path)

print("📁 Các file dự án tìm thấy để đính kèm:", candidate_files)
# Khi bắt đầu chat ở Cell 5, lượt hỏi đầu tiên sẽ tự động nạp các file này vào Sandbox (nếu chưa có Sandbox).

In [ ]:
# === CELL 5: CHAT LIÊN TIẾP VỚI AGENT ===
# Lệnh nhanh hỗ trợ trong lúc chat:
#   /upload <local_path> [remote_path]   -> Đẩy thêm file lên Sandbox
#   /files                               -> Liệt kê file trong Sandbox
#   /download <remote_path> [local_path] -> Tải file từ Sandbox về máy
#   /reset                               -> Xóa phiên cũ, tạo Sandbox mới
#   /exit hoặc /quit                     -> Thoát vòng lặp chat

print("💬 Bắt đầu phiên chat liên tiếp. Gõ /exit để dừng.")
first_turn = (session_state["env_id"] is None)

while True:
    try:
        user_msg = input("\n🧑 Bạn: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\n👋 Đã thoát chat.")
        break

    if not user_msg:
        continue
    if user_msg.lower() in ("/exit", "/quit", "exit", "quit"):
        print("👋 Đã kết thúc phiên chat.")
        break
    if user_msg.lower() == "/reset":
        session_state["env_id"] = None
        session_state["last_interaction_id"] = None
        first_turn = True
        print("🔄 Đã reset phiên.")
        continue
    if user_msg.lower() == "/files":
        list_sandbox_files()
        continue
    if user_msg.startswith("/upload "):
        parts = user_msg.split()
        if len(parts) >= 2:
            upload_to_sandbox(parts[1], parts[2] if len(parts) >= 3 else None)
        continue
    if user_msg.startswith("/download "):
        parts = user_msg.split()
        if len(parts) >= 2:
            download_from_sandbox(parts[1], parts[2] if len(parts) >= 3 else None)
        continue

    print("🤖 Agent đang xử lý...")
    attach = candidate_files if (first_turn and candidate_files) else None
    reply = ask_agent(user_msg, files=attach)
    first_turn = False
    print(f"\n🤖 Agent:\n{reply}")
    print(f"\n[Sandbox: {session_state['env_id']} | Interaction: {session_state['last_interaction_id']}]")

In [ ]:
# === CELL 6: KIỂM TRA & TẢI KẾT QUẢ VỀ (CHẠY KHI CẦN) ===
list_sandbox_files()
# Ví dụ tải file app.py đã được Agent sửa về thành app_updated.py:
# download_from_sandbox("app_updated.py", "../app_updated.py")